In [ ]:
#| default_exp pyparse

# Python module parsing

> Structured static views of Python modules, definitions, docstrings, literals, and `@patch` targets.

`parse_py` and `parse_py_file` build a lightweight `PyMod` tree from Python source without importing or executing it. Each node exposes its docstring and live `ast.Constant` node, child classes and functions in source order, dotted `qualname`, source span, and decorator names; `PyMod` adds top-level assignments and literal values.

In [ ]:
from fastcore.utils import *
import ast, inspect

In [ ]:
from fastcore.test import *
from tempfile import TemporaryDirectory

## Modules and definitions

In [ ]:
_DEF_TYPES = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)

def _doc_node(node):
    body = getattr(node, 'body', None)
    if not (body and isinstance(body[0], ast.Expr)): return None
    if isinstance(v := body[0].value, ast.Constant) and isinstance(v.value, str): return v

def decor_id(
    d:ast.AST # Decorator expression from `node.decorator_list`
)->str:
    "Name of decorator AST node `d`, whether bare, called, or dotted"
    if isinstance(d, ast.Call): d = d.func
    if isinstance(d, ast.Name): return d.id
    if isinstance(d, ast.Attribute): return d.attr
    return ''

`decor_id` normalises bare (`@patch`), called (`@patch(as_prop=True)`), and dotted (`@fastcore.patch`, `@x.setter`) decorator nodes to their final identifier, returning `''` for arbitrary expressions:

In [ ]:
test_eq([decor_id(d) for d in ast.parse('@a\n@b(1)\n@x.y\n@c[0]()\ndef f(): pass').body[0].decorator_list], ['a', 'b', 'y', ''])

In [ ]:
class PyDef:
    "Structured view of a Python module, class, or function AST node"
    def __init__(self, node, prefix='', src=None):
        self.node,self._src = node,src
        self.name = getattr(node, 'name', '')
        self.qualname = f'{prefix}.{self.name}' if prefix else self.name
        self.doc_node = _doc_node(node)
        self.doc = inspect.cleandoc(self.doc_node.value) if self.doc_node else ''
        self.decos = [decor_id(d) for d in getattr(node, 'decorator_list', ())]
        self.defs = L(PyDef(n, self.qualname, src) for n in node.body if isinstance(n, _DEF_TYPES))
        self.classes = self.defs.filter(lambda d: isinstance(d.node, ast.ClassDef))
        self.funcs = self.defs.filter(lambda d: not isinstance(d.node, ast.ClassDef))

    @property
    def methods(self): return self.funcs
    @property
    def lineno(self): return getattr(self.node, 'lineno', 1)
    @property
    def end_lineno(self): return getattr(self.node, 'end_lineno', None)
    @property
    def start_lineno(self):
        decs = getattr(self.node, 'decorator_list', ())
        return min((self.node.lineno, *(d.lineno for d in decs))) if hasattr(self.node, 'lineno') else 1
    @property
    def is_public(self): return bool(self.name) and not self.name.startswith('_')
    @property
    def all_defs(self): return L(o for d in self.defs for o in (d, *d.all_defs))
    @property
    def source(self):
        if self._src is None: return None
        if isinstance(self.node, ast.Module): return self._src
        return ''.join(self._src.splitlines(keepends=True)[self.start_lineno-1:self.end_lineno])
    def __repr__(self):
        k = type(self.node).__name__
        return f'{type(self).__name__}({k}, {self.qualname!r})' if self.qualname else f'{type(self).__name__}({k})'

`PyDef` wraps a module, class, or function AST node, collecting its direct child `defs` (`classes` and `funcs`/`methods`), cleaned `doc` and live `doc_node`, decorator names, and line span. `all_defs` walks direct bodies recursively in source order, and `source` slices from the first decorator through `end_lineno` when source text was supplied:

In [ ]:
d = PyDef(ast.parse('class C:\n    "Doc"\n    def m(self): pass'))
test_eq((d.classes[0].doc, d.all_defs.attrgot('qualname')), ('Doc', ['C', 'C.m']))

In [ ]:
def _assign_names(t):
    if isinstance(t, ast.Name): return [t.id]
    if isinstance(t, (ast.Tuple, ast.List)): return [nm for e in t.elts for nm in _assign_names(e)]
    if isinstance(t, ast.Starred): return _assign_names(t.value)
    return []

def _lit(node):
    try: return True, ast.literal_eval(node)
    except (ValueError, TypeError): return False, None

class PyMod(PyDef):
    "Structured view of a parsed Python module: docstring, definitions, assignments, and literals"
    def __init__(self, node, src=None):
        super().__init__(node, src=src)
        self.assigns,self.literals = {},{}
        for n in node.body:
            if isinstance(n, ast.Assign): targets,val = n.targets,n.value
            elif isinstance(n, ast.AnnAssign) and n.value is not None: targets,val = [n.target],n.value
            elif isinstance(n, ast.AugAssign): targets,val = [n.target],None
            else: continue
            ok,lit = _lit(val) if val is not None else (False, None)
            for t in targets:
                for nm in _assign_names(t):
                    self.assigns[nm] = n
                    if ok and isinstance(t, ast.Name): self.literals[nm] = lit
                    else: self.literals.pop(nm, None)

In [ ]:
def parse_py(
    src:str|ast.Module, # Python source string or pre-parsed `ast.Module`
    safe:bool=False, # Return `None` on parse error instead of raising?
    filename:str='<unknown>', # Filename passed to `ast.parse` for warnings and tracebacks
)->PyMod|None:
    "Parse Python source `src` (or wrap an `ast.Module`) into a `PyMod`"
    if isinstance(src, ast.Module): return PyMod(src)
    try: tree = ast.parse(src, filename=filename)
    except (SyntaxError, ValueError):
        if safe: return None
        raise
    return PyMod(tree, src=src)

`parse_py` walks direct module, class, and function bodies into `PyDef` nodes, keeping order and repeated names (such as `@property` / `@x.setter`) via `L`. Each node's `doc` is cleaned with `inspect.cleandoc`, while `doc_node` holds the live `ast.Constant` for line spans or AST mutation, and `source` slices from the first decorator through `end_lineno`:

In [ ]:
SRC = """"Demo module."
__docs__ = {'Docs': ['demo.core']}
__base_url__: str = 'https://example.com'
EMPTY = None
x = 1
x = compute()
__all__ = ['Box']
__all__ += ['run']
a, b = 1, 2
unassigned: int

class Box:
    "A container"
    @property
    def val(self): return 1
    @val.setter
    def val(self, v): pass
    async def fetch(self):
        "Load asynchronously"
        def _helper(): pass

def _priv(): pass

@fastcore.wraps(Box)
async def run():
    "Run it"
    pass
"""
m = parse_py(SRC)
test_eq(m.doc, 'Demo module.')
test_eq(m.doc_node.end_lineno, 1)
test_eq(m.defs.attrgot('qualname'), ['Box', '_priv', 'run'])
test_eq(m.all_defs.attrgot('qualname'), ['Box', 'Box.val', 'Box.val', 'Box.fetch', 'Box.fetch._helper', '_priv', 'run'])
test_eq([d.name for d in m.defs if d.is_public], ['Box', 'run'])
test_eq(m.classes[0].methods.attrgot('decos'), [['property'], ['setter'], []])
assert m.funcs[-1].source.startswith('@fastcore.wraps(Box)\nasync def run():')
m.all_defs

`PyMod.assigns` records every top-level assigned name (including annotated, augmented, and unpacked targets), while `PyMod.literals` keeps only values evaluable with `ast.literal_eval` and invalidates a name if a later statement reassigns or augments it non-literally:

In [ ]:
test_eq(list(m.assigns), ['__docs__', '__base_url__', 'EMPTY', 'x', '__all__', 'a', 'b'])
test_eq(m.literals, dict(__docs__={'Docs': ['demo.core']}, __base_url__='https://example.com', EMPTY=None))
assert parse_py(ast.parse('def f(): pass')).funcs[0].source is None
test_is(parse_py('def broken(', safe=True), None)
with expect_fail(SyntaxError): parse_py('def broken(')

In [ ]:
def parse_py_file(
    path:str|Path, # Path to a Python source file
    safe:bool=False, # Return `None` on read or parse error instead of raising?
)->PyMod|None:
    "Read and parse a Python file at `path` into a `PyMod`"
    try: src = Path(path).expanduser().read_text(encoding='utf-8')
    except (OSError, UnicodeDecodeError):
        if safe: return None
        raise
    return parse_py(src, safe=safe, filename=str(path))

`parse_py_file` reads a UTF-8 file path rather than source text, avoiding the trap where a string path such as `'pkg/mod.py'` would otherwise parse as a valid division expression:

In [ ]:
with TemporaryDirectory() as td:
    p = Path(td)/'sample.py'
    p.write_text('"Hi"\ndef f(): pass\n', encoding='utf-8')
    mf = parse_py_file(p)
    test_eq((mf.doc, mf.funcs[0].name), ('Hi', 'f'))
    (Path(td)/'bad.py').write_text('def (:\n', encoding='utf-8')
    test_is(parse_py_file(Path(td)/'bad.py', safe=True), None)
    test_is(parse_py_file(Path(td)/'missing.py', safe=True), None)
    (w := Path(td)/'warn.py').write_text('x = "\\s"\n', encoding='utf-8')
    import warnings
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always')
        parse_py_file(w)
    test_eq(ws[0].filename, str(w))

## `@patch` targets

`patch_targets` reads the target class names of `@patch` and `@patch_to` functions (sync or async), expanding union (`A | B`) and tuple annotations and handling positional-only `self`:

In [ ]:
def _target_names(node):
    if isinstance(node, ast.BinOp) and isinstance(node.op, ast.BitOr): return _target_names(node.left) + _target_names(node.right)
    if isinstance(node, (ast.Tuple, ast.List)): return [nm for e in node.elts for nm in _target_names(e)]
    return [ast.unparse(node).strip()]

def patch_targets(
    o # `PyDef` or AST node to inspect
)->list[str]:
    "Target class names for a `@patch` or `@patch_to` function, or `[]`"
    node = o.node if isinstance(o, PyDef) else o
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return []
    d = first(d for d in node.decorator_list if decor_id(d) in ('patch', 'patch_to'))
    if not d: return []
    if decor_id(d) == 'patch':
        args = node.args.posonlyargs or node.args.args
        ann = args[0].annotation if args else None
        return _target_names(ann) if ann else []
    if isinstance(d, ast.Call):
        arg = d.args[0] if d.args else first(k.value for k in d.keywords if k.arg == 'cls')
        return _target_names(arg) if arg else []
    return []

In [ ]:
PSRC = r'''
@patch
def a(self: Foo): pass
@fastcore.patch(as_prop=True)
async def b(self: A | mod.B, /): pass
@patch
def c(self: (X, Y)): pass
@patch_to(cls=Z)
def d(self): pass
@patch
def e(self): pass
def plain(): pass
class Cls: pass
'''
pm = parse_py(PSRC)
test_eq([patch_targets(d) for d in pm.defs], [['Foo'], ['A', 'mod.B'], ['X', 'Y'], ['Z'], [], [], []])
test_eq(patch_targets(pm.funcs[0].node), ['Foo'])

## Export -

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()